<a href="https://colab.research.google.com/github/bsheese/387/blob/main/21_training/21_1_Practical_Training/21_1_2_Training_a_Model_Well.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training in Practice: Part 3
## Training a Model Well

---

## What This Notebook Is About

21_1_1 showed that a network's validation loss is usually lowest partway through training, and that dropout and weight decay slow overfitting down. This notebook adds the last two pieces of a standard training recipe and then uses all of them together:

1. **Early stopping with a checkpoint**: save the weights whenever the validation loss reaches a new low, stop when it hasn't improved for a while, and go back to the best weights
2. **Saving and loading** a model's weights with `torch.save` and `load_state_dict`
3. **A learning-rate schedule**: large steps early in training, smaller steps later
4. **The full recipe** on Fashion-MNIST, checked once on the test set. Its test accuracy is the number 22_0's convolutional network has to beat.

In [ ]:
import copy
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'

DATA_URL = 'https://raw.githubusercontent.com/bsheese/387/main/data/'

def fetch(name):
    # Download a file from the course's data folder, unless it is already here.
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name

data = np.load(fetch('fashion10k.npz'))
CLASSES = list(data['classes'])

def prepare(images, labels):
    # 28 x 28 pixels from 0-255  ->  784 numbers from 0 to 1, one row per image
    X = torch.tensor(images.reshape(len(images), -1) / 255, dtype=torch.float32)
    return X.to(device), torch.tensor(labels, dtype=torch.long).to(device)

X_train, y_train = prepare(data['x_train'], data['y_train'])
X_val, y_val = prepare(data['x_val'], data['y_val'])
X_test, y_test = prepare(data['x_test'], data['y_test'])
loss_fn = nn.CrossEntropyLoss()
print('device:', device, '  training images:', len(X_train))

# The training loop from 21_0_1
def train_one_epoch(model, loader, optimizer):
    model.train()
    for x, y in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()

def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        scores = model(X)
        return loss_fn(scores, y).item(), (scores.argmax(dim=1) == y).float().mean().item()

def make_loader(X, y, batch_size=64):
    return torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=batch_size, shuffle=True)

def plot_history(histories):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    for label, h in histories.items():
        epochs = range(1, len(h['val_loss']) + 1)
        line, = ax1.plot(epochs, h['val_loss'], label=f'{label}: validation')
        ax1.plot(epochs, h['train_loss'], '--', color=line.get_color(), label=f'{label}: training')
        ax2.plot(epochs, h['val_acc'], label=f'{label}: validation', color=line.get_color())
        ax2.plot(epochs, h['train_acc'], '--', color=line.get_color(), label=f'{label}: training')
    ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend(fontsize=8)
    ax2.set_xlabel('epoch'); ax2.set_ylabel('accuracy'); ax2.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

The helper functions are the ones from 21_0_1 and 21_1_1.

---

## Section 1: Early Stopping with a Checkpoint

Here's the overfitting pattern from 21_1_1 again: the big network with no regularization, trained on all 10,000 training images for 30 epochs. The new `fit` function below records the same history as before, plus two things:

- Whenever the validation loss reaches a new lowest value, it saves a copy of the model's weights, called a **checkpoint**.
- If the validation loss hasn't improved for `patience` epochs in a row, it stops training early. Then it loads the checkpoint back into the model.

In [ ]:
def fit(model, optimizer, epochs, patience=None, scheduler=None, checkpoint_file='best_model.pt'):
    loader = make_loader(X_train, y_train)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
    best_loss, best_epoch = float('inf'), 0
    for epoch in range(1, epochs + 1):
        history['lr'].append(optimizer.param_groups[0]['lr'])   # the learning rate used this epoch
        train_one_epoch(model, loader, optimizer)
        if scheduler is not None:
            scheduler.step()                                     # Section 3
        for name, X, y in [('train', X_train, y_train), ('val', X_val, y_val)]:
            loss, acc = evaluate(model, X, y)
            history[f'{name}_loss'].append(loss)
            history[f'{name}_acc'].append(acc)

        if history['val_loss'][-1] < best_loss:                  # a new best: save a checkpoint
            best_loss, best_epoch = history['val_loss'][-1], epoch
            torch.save(model.state_dict(), checkpoint_file)
        elif patience is not None and epoch - best_epoch >= patience:
            print(f'stopping after epoch {epoch}: no improvement for {patience} epochs')
            break

    model.load_state_dict(torch.load(checkpoint_file, weights_only=True))   # go back to the best weights
    print(f'restored the weights from epoch {best_epoch} (validation loss {best_loss:.3f})')
    history['best_epoch'] = best_epoch
    return history

def big_network(dropout=0.0):
    return nn.Sequential(
        nn.Linear(784, 512), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(512, 512), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(512, 10),
    ).to(device)

The checkpoint is the model's `state_dict()`, the dictionary of all its weights from 20_1_1, written to a file with `torch.save`. Loading it back takes two steps: `torch.load` reads the dictionary from the file, and `model.load_state_dict` copies the numbers into the model. (`weights_only=True` tells `torch.load` to expect only tensors. A saved file can contain Python code that runs when it's loaded, so this is the safe way to load files, especially ones you didn't make yourself.)

Now train the unregularized network with a patience of 5 epochs:

In [ ]:
torch.manual_seed(0)
model = big_network()
stopped = fit(model, torch.optim.Adam(model.parameters(), lr=0.001), epochs=30, patience=5)

def plot_with_best(h, title):
    epochs = range(1, len(h['val_loss']) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, h['train_loss'], '--', label='training')
    plt.plot(epochs, h['val_loss'], label='validation')
    plt.axvline(h['best_epoch'], color='gray', linestyle=':', label=f"checkpoint (epoch {h['best_epoch']})")
    plt.xlabel('epoch'); plt.ylabel('loss'); plt.title(title); plt.legend()
    plt.show()

plot_with_best(stopped, 'Early stopping, patience 5')
print('validation accuracy of the restored model:', round(evaluate(model, X_val, y_val)[1], 3))

Training stopped once the validation loss had gone 5 epochs without a new low, and the model now holds the weights from the best epoch rather than the last one. Early stopping didn't make the network any less prone to overfitting. It just keeps the weights from before the overfitting got worse, and it saves the time that the remaining epochs would have taken.

Patience is a compromise. The validation loss bounces around from epoch to epoch, so stopping the first time it fails to improve would often stop too soon. A patience of a few epochs lets it recover from a bad epoch; a very long patience wastes time training a model you're going to throw away.

### The architecture must match

A state dictionary holds numbers, not the design of the network. To load one, you first build a model with exactly the same layers and sizes, and then copy the numbers in. If the shapes don't match, PyTorch refuses and lists the problems:

In [ ]:
wrong_size = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)
try:
    wrong_size.load_state_dict(torch.load('best_model.pt', weights_only=True))
except RuntimeError as e:
    print(str(e)[:400])

This is the same arrangement as 19_0: `pets4_head.pt` only stored the weights of an `nn.Linear(512, 4)`, so 19_0_2 had to build that layer before loading the file into it.

---

## Section 2: A Learning-Rate Schedule

20_0_3 showed the trade-off in choosing a learning rate: large steps make fast progress but bounce around near the bottom, and small steps settle in closely but make slow progress. A **learning-rate schedule** changes the learning rate during training to get some of both: larger steps early on, when the weights are far from good values, and smaller ones later, so the weights can settle.

A common schedule is **cosine annealing**, which lowers the learning rate along half of a cosine curve, from its starting value to nearly 0 at the last epoch. The scheduler is created from the optimizer, and `scheduler.step()` is called once at the end of each epoch (the `fit` function above already does this when it's given a scheduler).

In [ ]:
demo_optimizer = torch.optim.AdamW(big_network().parameters(), lr=0.001)
demo_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(demo_optimizer, T_max=30)
rates = []
for epoch in range(30):
    rates.append(demo_optimizer.param_groups[0]['lr'])
    demo_optimizer.step()         # in real training this is where an epoch of training happens
    demo_scheduler.step()

plt.figure(figsize=(6, 3.5))
plt.plot(range(1, 31), rates, marker='o')
plt.xlabel('epoch'); plt.ylabel('learning rate'); plt.title('Cosine annealing over 30 epochs')
plt.show()

(`T_max=30` is the number of epochs the curve is spread over. `optimizer.param_groups[0]['lr']` is where an optimizer keeps its current learning rate. The scheduler changes it there.)

Here is a regularized network, with dropout of 0.3 and weight decay of 0.05, trained for 30 epochs with a constant learning rate and with the cosine schedule:

In [ ]:
histories = {}
for label, use_schedule in [('constant learning rate', False), ('cosine schedule', True)]:
    torch.manual_seed(0)
    model = big_network(dropout=0.3)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30) if use_schedule else None
    print(label + ':')
    histories[label] = fit(model, optimizer, epochs=30, scheduler=scheduler, checkpoint_file=f'{label}.pt')

plt.figure(figsize=(7, 4))
for label, h in histories.items():
    plt.plot(range(1, 31), h['val_loss'], label=label)
plt.xlabel('epoch'); plt.ylabel('validation loss'); plt.legend()
plt.show()
for label, h in histories.items():
    print(f"{label:24} lowest validation loss {min(h['val_loss']):.3f}, final {h['val_loss'][-1]:.3f}")

The difference is modest. For the first several epochs the two runs are nearly the same, since the cosine learning rate is still close to its starting value. Later, the run with the schedule takes smaller steps and its validation loss settles lower and bounces around less, while the run with a constant rate keeps taking full-sized steps. A schedule rarely makes a dramatic difference on a problem like this one, but it costs nothing, and for long training runs on large networks it's standard practice. (Unit 23's language models use a related schedule that also starts with a short period of increasing learning rate, called warmup.)

---

## Section 3: The Full Recipe

Now everything together, as you might train any network:

- an architecture with dropout (0.3)
- AdamW with weight decay (0.05)
- a cosine learning-rate schedule
- early stopping with a checkpoint, so the final model has the weights with the lowest validation loss

Every choice above was made by looking at validation results, never test results.

In [ ]:
torch.manual_seed(0)
final_model = big_network(dropout=0.3)
optimizer = torch.optim.AdamW(final_model.parameters(), lr=0.001, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
final = fit(final_model, optimizer, epochs=30, patience=5, scheduler=scheduler, checkpoint_file='final_model.pt')
plot_with_best(final, 'The full recipe')

Compared with the unregularized network in Section 1, the gap between the training and validation losses grows much more slowly, and the best validation loss is lower.

Now the test set, once, as in 21_0_1:

In [ ]:
test_loss, test_acc = evaluate(final_model, X_test, y_test)
print(f'test accuracy: {test_acc:.3f}   (test loss {test_loss:.3f})')

with torch.no_grad():
    predicted = final_model(X_test).argmax(dim=1)
for k in range(10):
    mask = y_test == k
    print(f'{CLASSES[k]:12} {(predicted[mask] == k).float().mean().item():.2f}')

This network gets about 87% of the test images right, a couple of points better than 21_0_1's smaller network trained for 10 epochs with no regularization. The per-class results show the same weak spot as before: shirts are still the hardest class, followed by the other upper-body garments.

That leaves an honest question about where the remaining errors come from. Everything in this unit made the most of a network that sees each image as 784 separate numbers, with no idea which pixels are next to each other. More regularization and more epochs have stopped helping much. The next big improvement needs a network that can see shapes, which is what 22_0 builds. **This test accuracy is the number 22_0's convolutional network is compared against.**

### A checklist for training a network

These are the habits from unit 21, in the order you'd use them:

1. Split the data into training, validation and test sets before doing anything else, and leave the test set alone until the end (19_1_1).
2. Check that an untrained classifier's loss is near $-\log(1/\text{number of classes})$ (21_0_1).
3. Start with Adam or AdamW at a learning rate of 0.001, and watch both the training and validation curves (21_0_1).
4. If the validation loss starts rising while the training loss keeps falling, the network is overfitting. Get more data if you can; otherwise add dropout and weight decay (21_1_1).
5. Use early stopping with a checkpoint, so you keep the best weights rather than the last ones (this notebook).
6. Add a learning-rate schedule for longer runs (this notebook).
7. Measure on the test set once, at the end, and report that number.

---

## Summary

- **Early stopping** saves a **checkpoint** whenever the validation loss reaches a new low, stops after `patience` epochs without improvement, and restores the best weights. It keeps the weights from before overfitting got worse.
- `torch.save(model.state_dict(), file)` saves the weights. `model.load_state_dict(torch.load(file, weights_only=True))` loads them into a model that must have exactly the same architecture.
- A **learning-rate schedule** changes the learning rate during training. **Cosine annealing** lowers it smoothly from its starting value to nearly 0, which lets the weights settle; here it lowered the final validation loss a little.
- The full recipe (dropout, AdamW weight decay, cosine schedule, early stopping) trains an MLP that gets about 87% of Fashion-MNIST's test images right. Shirts and the other upper-body classes remain the hardest.
- The remaining errors come largely from the network's inability to see which pixels are neighbors, which is the problem convolutional networks address in 22_0.

**Next:** unit 22 starts with convolution: sliding a small grid of weights across an image, by hand and then in a network that replaces this MLP on the same Fashion-MNIST images.